# Concentrated Cleaning

In [2]:
import json
import spacy
from collections import Counter

# Load the medical-friendly model
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    """
    Standard cleaning pipeline:
    Tokenization, Lowercasing, Lemmatization, and Noise Removal.
    """
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.like_url
        and token.text.strip() != ""
        and token.is_alpha # Ensuring we keep only alphabetic words
    ]
    return " ".join(tokens)

In [3]:
# Load Documents
with open('all_docs_8.jsonl', 'r', encoding='utf-8') as f:
    docs = [json.loads(line) for line in f]

# Load Queries
with open('queries_8.jsonl', 'r', encoding='utf-8') as f:
    queries = [json.loads(line) for line in f]

# Prepare combined text strings
# Docs: Title + Abstract
doc_texts = [f"{d['title']} {d['abstract']}" for d in docs]

# Queries: Title + Need + Context
query_texts = [f"{q['title']} {q['need']} {q['context']}" for q in queries]

print(f"Total documents to process: {len(doc_texts)}")
print(f"Total queries to process: {len(query_texts)}")

Total documents to process: 3000
Total queries to process: 5


In [6]:
print("Cleaning documents")
cleaned_docs = [clean_text(txt) for txt in doc_texts]

print("Cleaning queries")
cleaned_queries = [clean_text(txt) for txt in query_texts]

# Example of cleaned output
print(f"Original Query 1: {query_texts[0][:100]}")
print(f"Cleaned Query 1: {cleaned_queries[0]}")

Cleaning documents
Cleaning queries
Original Query 1: Generating transgenic mice Find protocols for generating transgenic mice. Determine protocols to gen
Cleaned Query 1: generate transgenic mouse find protocol generate transgenic mouse determine protocol generate transgenic mouse have single copy gene interest specific location


# Binary Dictionary

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the vectorizer for Binary Search (0/1)
binary_vectorizer = CountVectorizer(binary=True)

# Fit on documents to build the global dictionary V, and transform docs to vectors
X_binary = binary_vectorizer.fit_transform(cleaned_docs)

# Transform queries using the same dictionary V
Q_binary = binary_vectorizer.transform(cleaned_queries)

# Get the dictionary (Vocabulary)
vocabulary = binary_vectorizer.get_feature_names_out()

print(f"Dictionary size (|V|): {len(vocabulary)}")
print(f"Document matrix shape: {X_binary.shape}") # (num_docs, |V|)
print(f"Query matrix shape: {Q_binary.shape}")   # (num_queries, |V|)

Dictionary size (|V|): 18574
Document matrix shape: (3000, 18574)
Query matrix shape: (5, 18574)


# Frequency Dictionary

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Frequency / Co-occurrence Vectors
# We use CountVectorize to get the actual count of each word in the document.
freq_vectorizer = CountVectorizer()

# Fit and transform the cleaned documents
X_freq = freq_vectorizer.fit_transform(cleaned_docs)
# Transform the queries using the same vocabulary
Q_freq = freq_vectorizer.transform(cleaned_queries)

print(f"Frequency Matrix (Docs): {X_freq.shape}")
print(f"Frequency Matrix (Queries): {Q_freq.shape}")

Frequency Matrix (Docs): (3000, 18574)
Frequency Matrix (Queries): (5, 18574)


# TF-IDF Dictionary

In [9]:
# TF-IDF Vectors
# TfidfVectorizer calculates both the Term Frequency (TF)
# and the Inverse Document Frequency (IDF) automatically.
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the cleaned documents
X_tfidf = tfidf_vectorizer.fit_transform(cleaned_docs)
# Transform the queries using the same global IDF values
Q_tfidf = tfidf_vectorizer.transform(cleaned_queries)

print(f"TF-IDF Matrix (Docs): {X_tfidf.shape}")
print(f"TF-IDF Matrix (Queries): {Q_tfidf.shape}")

idf_vector = tfidf_vectorizer.idf_
print(f"Global IDF vector size: {len(idf_vector)}")

TF-IDF Matrix (Docs): (3000, 18574)
TF-IDF Matrix (Queries): (5, 18574)
Global IDF vector size: 18574


# .txt Vector Files

In [10]:
import numpy as np

# Function to save vectors in the specific format: [x1, x2, ..., xn]
def save_vectors_to_txt(matrix, file_name, limit=None):
    """
    Saves a sparse matrix as a text file with vectors in [x1, x2, ...] format.
    """
    # Convert to dense format to handle all values including zeros
    dense_matrix = matrix.todense()

    if limit:
        dense_matrix = dense_matrix[:limit]

    with open(file_name, 'w', encoding='utf-8') as f:
        for i in range(dense_matrix.shape[0]):
            # Convert row to a list of numbers
            row_list = dense_matrix[i].tolist()[0]
            # Write to file in the requested format with brackets and commas
            f.write(str(row_list) + "\n")
    print(f"Saved vectors to {file_name}")

# Save the Dictionary (Vocabulary) in format [v1, v2, ..., vn]
with open('vocabulary.txt', 'w', encoding='utf-8') as f:
    vocab_list = list(vocabulary)
    f.write(str(vocab_list) + "\n")
print("Saved vocabulary to vocabulary.txt")

# Save 10 arbitrary document vectors (using TF-IDF as an example)
# You can repeat this for binary or freq matrices if needed by changing the input matrix
save_vectors_to_txt(X_tfidf, 'doc_vectors_tfidf.txt', limit=10)

# Save all 5 query vectors
save_vectors_to_txt(Q_tfidf, 'query_vectors_tfidf.txt')

Saved vocabulary to vocabulary.txt
Saved vectors to doc_vectors_tfidf.txt
Saved vectors to query_vectors_tfidf.txt


In [11]:
# Saving Binary Vectors (0/1)
# Saving 10 arbitrary document vectors for Binary
save_vectors_to_txt(X_binary, 'doc_vectors_binary.txt', limit=10)
# Saving all 5 query vectors for Binary
save_vectors_to_txt(Q_binary, 'query_vectors_binary.txt')

# Saving Frequency Vectors (Counts)
# Saving 10 arbitrary document vectors for Frequency
save_vectors_to_txt(X_freq, 'doc_vectors_freq.txt', limit=10)
# Saving all 5 query vectors for Frequency
save_vectors_to_txt(Q_freq, 'query_vectors_freq.txt')

print("All Binary and Frequency files have been saved successfully.")

Saved vectors to doc_vectors_binary.txt
Saved vectors to query_vectors_binary.txt
Saved vectors to doc_vectors_freq.txt
Saved vectors to query_vectors_freq.txt
All Binary and Frequency files have been saved successfully.
